In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
import matplotlib.pyplot as plt

data = pd.read_csv("../data/netflix_data.csv", parse_dates=['Date'], index_col='Date')
close_prices = data['Close'].values.reshape(-1,1)

scaler = MinMaxScaler(feature_range=(0,1))
scaled_data = scaler.fit_transform(close_prices)

time_step = 50
X, y = [], []
for i in range(time_step, len(scaled_data)):
    X.append(scaled_data[i-time_step:i,0])
    y.append(scaled_data[i,0])
X, y = np.array(X), np.array(y)
X = X.reshape(X.shape[0], X.shape[1], 1)

train_size = int(len(X)*0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

model = Sequential()
model.add(LSTM(50, return_sequences=True, input_shape=(time_step,1)))
model.add(LSTM(50))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mean_squared_error')

model.fit(X_train, y_train, epochs=20, batch_size=32, validation_data=(X_test, y_test))

predicted = model.predict(X_test)
predicted_prices = scaler.inverse_transform(predicted)

plt.figure(figsize=(12,6))
plt.plot(data.index[-len(predicted_prices):], predicted_prices, label='Predicted')
plt.plot(data.index[-len(predicted_prices):], close_prices[-len(predicted_prices):], label='Actual')
plt.title("Netflix Actual vs Predicted Prices")
plt.legend()
plt.show()
